In [2]:
import csv
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold

# -------------------------------------------------------------------------
# Step 1: Pre-calculate Train Statistics for Leak-Free Imputation
# -------------------------------------------------------------------------
train_raw = pd.read_csv("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/student_dataset.csv")

numeric_cols = [
    "ambient_temp",
    "process_temp",
    "coolant_flow",
    "shaft_speed",
    "drive_torque",
    "power_load",
    "component_wear",
    "thermal_stress",
    "mechanical_load",
    "vibration_index",
]

# Baseline medians and mode
imputation_medians = train_raw[numeric_cols].median().to_dict()
unit_mode = train_raw["unit_type"].mode()[0]
unit_type_mapping = {"S": 0, "M": 1, "H": 2}


# -------------------------------------------------------------------------
# Step 2: Row-by-Row Preprocessing & Domain Feature Engineering
# -------------------------------------------------------------------------
def process_dataset_row_by_row(
    file_path: str, is_train: bool = True
) -> pd.DataFrame:
    processed_rows = []

    with open(file_path, mode="r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            clean_record = {"unit_id": row.get("unit_id", "")}

            # 1. Categorical parsing & encoding with mode imputation
            raw_unit = row.get("unit_type", "").strip().upper()
            clean_unit = (
                raw_unit if raw_unit in unit_type_mapping else unit_mode
            )
            clean_record["unit_type_enc"] = unit_type_mapping[clean_unit]

            # 2. Continuous feature casting & median imputation
            for col in numeric_cols:
                val_str = row.get(col, "").strip()
                if val_str == "" or val_str.lower() == "nan":
                    clean_record[col] = float(imputation_medians[col])
                else:
                    try:
                        clean_record[col] = float(val_str)
                    except ValueError:
                        clean_record[col] = float(imputation_medians[col])

            # 3. Domain feature engineering
            clean_record["temp_diff"] = (
                clean_record["process_temp"] - clean_record["ambient_temp"]
            )
            clean_record["calculated_power"] = (
                clean_record["shaft_speed"] * clean_record["drive_torque"]
            ) / 9550.0
            clean_record["wear_vibe_ratio"] = (
                clean_record["component_wear"] * clean_record["vibration_index"]
            )
            clean_record["thermal_coolant_ratio"] = clean_record[
                "thermal_stress"
            ] / (clean_record["coolant_flow"] + 1e-5)

            # 4. Target label extraction
            if is_train and "condition" in row and row["condition"] != "":
                clean_record["condition"] = int(float(row["condition"]))

            processed_rows.append(clean_record)

    return pd.DataFrame(processed_rows)


print("Preprocessing training and testing datasets row-by-row...")
train_df = process_dataset_row_by_row("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/student_dataset.csv", is_train=True)
test_df = process_dataset_row_by_row("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/test.csv", is_train=False)

feature_columns = [
    c for c in train_df.columns if c not in ["unit_id", "condition"]
]
X = train_df[feature_columns].copy()
y = train_df["condition"].values
X_test = test_df[feature_columns].copy()

print(f"Features: {feature_columns}")
print(f"Train Shape: {X.shape} | Test Shape: {X_test.shape}")

# -------------------------------------------------------------------------
# Step 3: Stratified 5-Fold Soft-Voting Ensemble (HistGBM + RF)
# -------------------------------------------------------------------------
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_probabilities = np.zeros((len(X), 3))
test_probabilities = np.zeros((len(X_test), 3))

print("\nStarting Cross-Validation & Ensemble Training...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_train_f, y_train_f = X.iloc[train_idx], y[train_idx]
    X_val_f, y_val_f = X.iloc[val_idx], y[val_idx]

    # Model 1: High-capacity Gradient Booster
    model_hgb_deep = HistGradientBoostingClassifier(
        max_iter=350,
        learning_rate=0.04,
        max_leaf_nodes=45,
        min_samples_leaf=20,
        random_state=42 + fold,
    )

    # Model 2: Conservative L2-Regularized Gradient Booster
    model_hgb_reg = HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.06,
        max_leaf_nodes=25,
        l2_regularization=1.0,
        random_state=100 + fold,
    )

    # Model 3: Variance-Reducing Random Forest
    model_rf = RandomForestClassifier(
        n_estimators=250,
        max_depth=16,
        min_samples_leaf=3,
        random_state=42 + fold,
        n_jobs=-1,
    )

    # Fit all three models
    model_hgb_deep.fit(X_train_f, y_train_f)
    model_hgb_reg.fit(X_train_f, y_train_f)
    model_rf.fit(X_train_f, y_train_f)

    # Soft-voting probability weighting
    val_probs = (
        0.45 * model_hgb_deep.predict_proba(X_val_f)
        + 0.35 * model_hgb_reg.predict_proba(X_val_f)
        + 0.20 * model_rf.predict_proba(X_val_f)
    )
    oof_probabilities[val_idx] = val_probs

    # Accumulate fold predictions for the test set
    test_probs = (
        0.45 * model_hgb_deep.predict_proba(X_test)
        + 0.35 * model_hgb_reg.predict_proba(X_test)
        + 0.20 * model_rf.predict_proba(X_test)
    )
    test_probabilities += test_probs / N_SPLITS

    fold_acc = accuracy_score(y_val_f, np.argmax(val_probs, axis=1))
    print(f"  Fold {fold} — Validation Accuracy: {fold_acc:.4f}")

# -------------------------------------------------------------------------
# Step 4: Model Evaluation
# -------------------------------------------------------------------------
oof_predictions = np.argmax(oof_probabilities, axis=1)
total_acc = accuracy_score(y, oof_predictions)
total_f1 = f1_score(y, oof_predictions, average="macro")

print("\n" + "=" * 50)
print(f"OVERALL OUT-OF-FOLD ACCURACY: {total_acc:.4f}")
print(f"OVERALL OUT-OF-FOLD MACRO F1: {total_f1:.4f}")
print("=" * 50)
print("\nClassification Report:\n", classification_report(y, oof_predictions))

# -------------------------------------------------------------------------
# Step 5: Generate Kaggle Submission File
# -------------------------------------------------------------------------
final_predictions = np.argmax(test_probabilities, axis=1)

submission = pd.DataFrame(
    {"unit_id": test_df["unit_id"], "condition": final_predictions}
)

submission.to_csv("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/submission.csv", index=False)
print("File successfully created: 'submission.csv'")
print("\nSubmission preview:")
print(submission.head(10))

Preprocessing training and testing datasets row-by-row...
Features: ['unit_type_enc', 'ambient_temp', 'process_temp', 'coolant_flow', 'shaft_speed', 'drive_torque', 'power_load', 'component_wear', 'thermal_stress', 'mechanical_load', 'vibration_index', 'temp_diff', 'calculated_power', 'wear_vibe_ratio', 'thermal_coolant_ratio']
Train Shape: (16000, 15) | Test Shape: (4000, 15)

Starting Cross-Validation & Ensemble Training...
  Fold 1 — Validation Accuracy: 0.8825
  Fold 2 — Validation Accuracy: 0.8825
  Fold 3 — Validation Accuracy: 0.8878
  Fold 4 — Validation Accuracy: 0.8819
  Fold 5 — Validation Accuracy: 0.8831

OVERALL OUT-OF-FOLD ACCURACY: 0.8836
OVERALL OUT-OF-FOLD MACRO F1: 0.8731

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.94      0.91      8000
           1       0.88      0.83      0.86      4000
           2       0.88      0.82      0.85      4000

    accuracy                           0.88     16000
   

In [3]:
import csv
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold

# -------------------------------------------------------------------------
# Step 1: Pre-calculate Baselines & Unit Group Statistics (Leak-Free)
# -------------------------------------------------------------------------
train_raw = pd.read_csv("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/student_dataset.csv")

numeric_cols = [
    "ambient_temp",
    "process_temp",
    "coolant_flow",
    "shaft_speed",
    "drive_torque",
    "power_load",
    "component_wear",
    "thermal_stress",
    "mechanical_load",
    "vibration_index",
]

# Global medians and mode
imputation_medians = train_raw[numeric_cols].median().to_dict()
unit_mode = train_raw["unit_type"].mode()[0]
unit_type_mapping = {"S": 0, "M": 1, "H": 2}

# Compute operating-envelope means and stds per unit_type from training data
group_cols = [
    "process_temp",
    "shaft_speed",
    "drive_torque",
    "power_load",
    "thermal_stress",
    "mechanical_load",
    "component_wear",
]
group_stats = {}
for u in ["S", "M", "H"]:
    subset = train_raw[train_raw["unit_type"] == u]
    group_stats[u] = {
        "mean": subset[group_cols].mean().to_dict(),
        "std": (subset[group_cols].std() + 1e-6).to_dict(),
    }


# -------------------------------------------------------------------------
# Step 2: Row-by-Row Preprocessing & Advanced Domain Feature Engineering
# -------------------------------------------------------------------------
def process_dataset_row_by_row(
    file_path: str, is_train: bool = True
) -> pd.DataFrame:
    processed_rows = []

    with open(file_path, mode="r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            clean = {"unit_id": row.get("unit_id", "")}

            # 1. Parse unit type
            raw_unit = row.get("unit_type", "").strip().upper()
            unit_val = raw_unit if raw_unit in unit_type_mapping else unit_mode
            clean["unit_type"] = unit_val
            clean["unit_type_enc"] = unit_type_mapping[unit_val]

            # 2. Impute and cast numeric sensors
            for col in numeric_cols:
                v = row.get(col, "").strip()
                if v == "" or v.lower() == "nan":
                    clean[col] = float(imputation_medians[col])
                else:
                    try:
                        clean[col] = float(v)
                    except ValueError:
                        clean[col] = float(imputation_medians[col])

            # 3. Thermodynamic & Thermal dissipation features
            clean["delta_temp"] = clean["process_temp"] - clean["ambient_temp"]
            clean["cooling_efficiency"] = clean["delta_temp"] / (
                clean["coolant_flow"] + 1e-5
            )
            clean["thermal_to_coolant"] = clean["thermal_stress"] / (
                clean["coolant_flow"] + 1e-5
            )

            # 4. Mechanical & Power balance features
            clean["theoretical_power"] = (
                clean["shaft_speed"] * clean["drive_torque"]
            ) / 9550.0
            clean["power_discrepancy"] = (
                clean["power_load"] - clean["theoretical_power"]
            )
            clean["power_ratio"] = clean["power_load"] / (
                clean["theoretical_power"] + 1e-5
            )

            # 5. Wear & Vibration dynamic strain features
            clean["wear_vibe"] = (
                clean["component_wear"] * clean["vibration_index"]
            )
            clean["wear_vibe_sq"] = clean["component_wear"] * (
                clean["vibration_index"] ** 2
            )
            clean["vibe_speed_energy"] = clean["vibration_index"] * (
                (clean["shaft_speed"] / 1000.0) ** 2
            )
            clean["mechanical_vibe"] = (
                clean["mechanical_load"] * clean["vibration_index"]
            )
            clean["thermal_mechanical_ratio"] = clean["thermal_stress"] / (
                clean["mechanical_load"] + 1e-5
            )
            clean["total_stress"] = (
                clean["thermal_stress"] + clean["mechanical_load"]
            )

            # 6. Unit-type normalized relative features (Z-score & delta relative to unit group)
            u_mean = group_stats[unit_val]["mean"]
            u_std = group_stats[unit_val]["std"]
            for col in group_cols:
                clean[f"{col}_diff_unit"] = clean[col] - u_mean[col]
                clean[f"{col}_z_unit"] = (clean[col] - u_mean[col]) / u_std[col]

            # 7. Extract target
            if is_train and "condition" in row and row["condition"] != "":
                clean["condition"] = int(float(row["condition"]))

            processed_rows.append(clean)

    return pd.DataFrame(processed_rows)


print("Preprocessing data with domain physics and group normalizations...")
train_df = process_dataset_row_by_row("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/student_dataset.csv", is_train=True)
test_df = process_dataset_row_by_row("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/test.csv", is_train=False)

drop_cols = ["unit_id", "unit_type", "condition"]
feature_cols = [c for c in train_df.columns if c not in drop_cols]
X = train_df[feature_cols].copy()
y = train_df["condition"].values
X_test = test_df[feature_cols].copy()

print(f"Features: {len(feature_cols)} active variables.")
print(f"Train Shape: {X.shape} | Test Shape: {X_test.shape}")

# -------------------------------------------------------------------------
# Step 3: Stratified 5-Fold Tri-Model Ensemble (HistGBM + RF + ExtraTrees)
# -------------------------------------------------------------------------
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_probs = np.zeros((len(X), 3))
test_probs = np.zeros((len(X_test), 3))

print("\nStarting Cross-Validation Ensemble Training...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_tr, y_tr = X.iloc[train_idx], y[train_idx]
    X_va, y_va = X.iloc[val_idx], y[val_idx]

    # Model 1: High-capacity HistGBM (captures multi-sensor non-linear thresholds)
    m1 = HistGradientBoostingClassifier(
        max_iter=400,
        learning_rate=0.04,
        max_leaf_nodes=40,
        min_samples_leaf=20,
        l2_regularization=0.5,
        random_state=42 + fold,
    )

    # Model 2: Conservative Regularized HistGBM (smooths edge predictions)
    m2 = HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=25,
        min_samples_leaf=30,
        l2_regularization=2.0,
        random_state=100 + fold,
    )

    # Model 3: Tuned Random Forest (decorrelated variance reduction)
    m3 = RandomForestClassifier(
        n_estimators=300,
        max_depth=16,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42 + fold,
        n_jobs=-1,
    )

    # Model 4: ExtraTrees (extreme random splits to cancel sensor noise)
    m4 = ExtraTreesClassifier(
        n_estimators=250,
        max_depth=18,
        min_samples_leaf=2,
        random_state=77 + fold,
        n_jobs=-1,
    )

    # Fit all models
    m1.fit(X_tr, y_tr)
    m2.fit(X_tr, y_tr)
    m3.fit(X_tr, y_tr)
    m4.fit(X_tr, y_tr)

    # Calibrated probability blend
    val_pred = (
        0.40 * m1.predict_proba(X_va)
        + 0.30 * m2.predict_proba(X_va)
        + 0.15 * m3.predict_proba(X_va)
        + 0.15 * m4.predict_proba(X_va)
    )
    oof_probs[val_idx] = val_pred

    test_pred = (
        0.40 * m1.predict_proba(X_test)
        + 0.30 * m2.predict_proba(X_test)
        + 0.15 * m3.predict_proba(X_test)
        + 0.15 * m4.predict_proba(X_test)
    )
    test_probs += test_pred / N_SPLITS

    fold_acc = accuracy_score(y_va, np.argmax(val_pred, axis=1))
    print(f"  Fold {fold} — Validation Accuracy: {fold_acc:.4f}")

# -------------------------------------------------------------------------
# Step 4: Out-Of-Fold Evaluation
# -------------------------------------------------------------------------
oof_preds = np.argmax(oof_probs, axis=1)
total_acc = accuracy_score(y, oof_preds)
macro_f1 = f1_score(y, oof_preds, average="macro")

print("\n" + "=" * 55)
print(f"NEW OVERALL OOF ACCURACY: {total_acc:.4f}")
print(f"NEW OVERALL MACRO F1:     {macro_f1:.4f}")
print("=" * 55)
print("\nClassification Report:\n", classification_report(y, oof_preds))

# -------------------------------------------------------------------------
# Step 5: Generate Kaggle Submission File
# -------------------------------------------------------------------------
final_test_classes = np.argmax(test_probs, axis=1)

sub = pd.DataFrame(
    {"unit_id": test_df["unit_id"], "condition": final_test_classes}
)

sub.to_csv("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/submission_improved.csv", index=False)
print(
    "\nSuccessfully generated 'submission_improved.csv' ready for upload on Kaggle!"
)
print("Predicted Test Class Distribution:")
print(sub["condition"].value_counts(normalize=True))

Preprocessing data with domain physics and group normalizations...
Features: 37 active variables.
Train Shape: (16000, 37) | Test Shape: (4000, 37)

Starting Cross-Validation Ensemble Training...
  Fold 1 — Validation Accuracy: 0.8872
  Fold 2 — Validation Accuracy: 0.8862
  Fold 3 — Validation Accuracy: 0.8900
  Fold 4 — Validation Accuracy: 0.8888
  Fold 5 — Validation Accuracy: 0.8875

NEW OVERALL OOF ACCURACY: 0.8879
NEW OVERALL MACRO F1:     0.8776

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.94      0.92      8000
           1       0.88      0.83      0.86      4000
           2       0.89      0.83      0.86      4000

    accuracy                           0.89     16000
   macro avg       0.89      0.87      0.88     16000
weighted avg       0.89      0.89      0.89     16000


Successfully generated 'submission_improved.csv' ready for upload on Kaggle!
Predicted Test Class Distribution:
condition
0    0.52825


In [4]:
import csv
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold

# -------------------------------------------------------------------------
# 1. Leak-Free Global Statistics & Operating Envelopes
# -------------------------------------------------------------------------
train_raw = pd.read_csv("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/student_dataset.csv")

num_cols = [
    "ambient_temp",
    "process_temp",
    "coolant_flow",
    "shaft_speed",
    "drive_torque",
    "power_load",
    "component_wear",
    "thermal_stress",
    "mechanical_load",
    "vibration_index",
]

medians = train_raw[num_cols].median().to_dict()
unit_mode = train_raw["unit_type"].mode()[0]
unit_type_mapping = {"S": 0, "M": 1, "H": 2}

group_cols = [
    "process_temp",
    "shaft_speed",
    "drive_torque",
    "power_load",
    "thermal_stress",
    "mechanical_load",
    "component_wear",
]

group_stats = {}
for u in ["S", "M", "H"]:
    sub = train_raw[train_raw["unit_type"] == u]
    group_stats[u] = {
        "mean": sub[group_cols].mean().to_dict(),
        "std": (sub[group_cols].std() + 1e-6).to_dict(),
    }


# -------------------------------------------------------------------------
# 2. Physics & Failure-Regime Feature Engineering
# -------------------------------------------------------------------------
def process_data(file_path: str, is_train: bool = True) -> pd.DataFrame:
    rows = []
    with open(file_path, mode="r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for r in reader:
            clean = {"unit_id": r.get("unit_id", "")}

            # Categorical unit type
            raw_u = r.get("unit_type", "").strip().upper()
            u_val = raw_u if raw_u in unit_type_mapping else unit_mode
            clean["unit_type"] = u_val
            clean["unit_type_enc"] = unit_type_mapping[u_val]

            # Sensor imputations
            for col in num_cols:
                v = r.get(col, "").strip()
                if v == "" or v.lower() == "nan":
                    clean[col] = float(medians[col])
                else:
                    try:
                        clean[col] = float(v)
                    except ValueError:
                        clean[col] = float(medians[col])

            # Thermal & Dissipation
            clean["delta_temp"] = clean["process_temp"] - clean["ambient_temp"]
            clean["cooling_efficiency"] = clean["delta_temp"] / (
                clean["coolant_flow"] + 1e-5
            )
            clean["thermal_to_coolant"] = clean["thermal_stress"] / (
                clean["coolant_flow"] + 1e-5
            )

            # Mechanical & Overstrain (Key failure formula)
            clean["theoretical_power"] = (
                clean["shaft_speed"] * clean["drive_torque"]
            ) / 9550.0
            clean["power_discrepancy"] = (
                clean["power_load"] - clean["theoretical_power"]
            )
            clean["power_ratio"] = clean["power_load"] / (
                clean["theoretical_power"] + 1e-5
            )
            clean["overstrain_index"] = (
                clean["component_wear"] * clean["drive_torque"]
            )

            # Wear & Dynamic Vibration Strain
            clean["wear_vibe"] = (
                clean["component_wear"] * clean["vibration_index"]
            )
            clean["wear_vibe_sq"] = clean["component_wear"] * (
                clean["vibration_index"] ** 2
            )
            clean["vibe_speed_energy"] = clean["vibration_index"] * (
                (clean["shaft_speed"] / 1000.0) ** 2
            )
            clean["mechanical_vibe"] = (
                clean["mechanical_load"] * clean["vibration_index"]
            )
            clean["thermal_mechanical_ratio"] = clean["thermal_stress"] / (
                clean["mechanical_load"] + 1e-5
            )
            clean["total_stress"] = (
                clean["thermal_stress"] + clean["mechanical_load"]
            )

            # Non-linear failure boundary flags
            clean["flag_high_temp"] = float(clean["process_temp"] > 69.15)
            clean["flag_high_wear"] = float(clean["component_wear"] > 185.0)
            clean["flag_high_mech"] = float(clean["mechanical_load"] > 100.0)
            clean["flag_high_vibe"] = float(clean["vibration_index"] > 2.1)

            # Group relative normalizations
            u_m = group_stats[u_val]["mean"]
            u_s = group_stats[u_val]["std"]
            for col in group_cols:
                clean[f"{col}_diff_unit"] = clean[col] - u_m[col]
                clean[f"{col}_z_unit"] = (clean[col] - u_m[col]) / u_s[col]

            if is_train and "condition" in r and r["condition"] != "":
                clean["condition"] = int(float(r["condition"]))

            rows.append(clean)
    return pd.DataFrame(rows)


print("Preprocessing and engineering features...")
train_df = process_data("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/student_dataset.csv", is_train=True)
test_df = process_data("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/test.csv", is_train=False)

feats = [
    c
    for c in train_df.columns
    if c not in ["unit_id", "unit_type", "condition"]
]
X = train_df[feats].copy()
y = train_df["condition"].values
X_test = test_df[feats].copy()

print(f"Features: {len(feats)} dimensions | Total samples: {len(X)}")

# -------------------------------------------------------------------------
# 3. 10-Fold Stratified Multi-Model Blending
# -------------------------------------------------------------------------
N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_probs = np.zeros((len(X), 3))
test_probs = np.zeros((len(X_test), 3))

print("\nTraining 10-Fold Tri-Model Pipeline...")

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_tr, y_tr = X.iloc[trn_idx], y[trn_idx]
    X_va, y_va = X.iloc[val_idx], y[val_idx]

    # Model 1: Deep HistGBM
    m1 = HistGradientBoostingClassifier(
        max_iter=450,
        learning_rate=0.035,
        max_leaf_nodes=42,
        min_samples_leaf=18,
        l2_regularization=0.8,
        random_state=42 + fold,
    )

    # Model 2: Wider, Regularized HistGBM
    m2 = HistGradientBoostingClassifier(
        max_iter=350,
        learning_rate=0.045,
        max_leaf_nodes=31,
        min_samples_leaf=25,
        l2_regularization=1.5,
        random_state=100 + fold,
    )

    # Model 3: Tuned Random Forest
    m3 = RandomForestClassifier(
        n_estimators=300,
        max_depth=17,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42 + fold,
        n_jobs=-1,
    )

    # Model 4: ExtraTrees Classifier
    m4 = ExtraTreesClassifier(
        n_estimators=300,
        max_depth=19,
        min_samples_leaf=2,
        random_state=77 + fold,
        n_jobs=-1,
    )

    m1.fit(X_tr, y_tr)
    m2.fit(X_tr, y_tr)
    m3.fit(X_tr, y_tr)
    m4.fit(X_tr, y_tr)

    val_blend = (
        0.35 * m1.predict_proba(X_va)
        + 0.25 * m2.predict_proba(X_va)
        + 0.20 * m3.predict_proba(X_va)
        + 0.20 * m4.predict_proba(X_va)
    )
    oof_probs[val_idx] = val_blend

    test_blend = (
        0.35 * m1.predict_proba(X_test)
        + 0.25 * m2.predict_proba(X_test)
        + 0.20 * m3.predict_proba(X_test)
        + 0.20 * m4.predict_proba(X_test)
    )
    test_probs += test_blend / N_SPLITS

    fold_acc = accuracy_score(y_va, np.argmax(val_blend, axis=1))
    print(f"  Fold {fold:02d} Accuracy: {fold_acc:.4f}")

# -------------------------------------------------------------------------
# 4. Out-of-Fold Threshold / Probability Calibration
# -------------------------------------------------------------------------
def loss_objective(weights):
    scaled = oof_probs * np.array(weights)
    return -accuracy_score(y, np.argmax(scaled, axis=1))

res = minimize(
    loss_objective,
    [1.0, 1.0, 1.0],
    method="Nelder-Mead",
    options={"maxiter": 500},
)
opt_weights = res.x / np.max(res.x)

raw_oof_acc = accuracy_score(y, np.argmax(oof_probs, axis=1))
calibrated_oof_preds = np.argmax(oof_probs * opt_weights, axis=1)
calibrated_oof_acc = accuracy_score(y, calibrated_oof_preds)

print("\n" + "=" * 55)
print(f"Raw 10-Fold OOF Accuracy:        {raw_oof_acc:.5f}")
print(f"Calibrated 10-Fold OOF Accuracy: {calibrated_oof_acc:.5f}")
print(f"Optimal Class Multipliers:       {np.round(opt_weights, 4)}")
print("=" * 55)

# -------------------------------------------------------------------------
# 5. Export Calibrated Kaggle Submission
# -------------------------------------------------------------------------
final_test_preds = np.argmax(test_probs * opt_weights, axis=1)

sub = pd.DataFrame(
    {"unit_id": test_df["unit_id"], "condition": final_test_preds}
)

sub.to_csv("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/submission_90.csv", index=False)
print("\nGenerated 'submission_90.csv' for submission.")
print("Predicted Class Proportions:")
print(sub["condition"].value_counts(normalize=True))

Preprocessing and engineering features...
Features: 42 dimensions | Total samples: 16000

Training 10-Fold Tri-Model Pipeline...
  Fold 01 Accuracy: 0.8994
  Fold 02 Accuracy: 0.8712
  Fold 03 Accuracy: 0.8881
  Fold 04 Accuracy: 0.8894
  Fold 05 Accuracy: 0.8900
  Fold 06 Accuracy: 0.8956
  Fold 07 Accuracy: 0.8912
  Fold 08 Accuracy: 0.8788
  Fold 09 Accuracy: 0.8806
  Fold 10 Accuracy: 0.8938

Raw 10-Fold OOF Accuracy:        0.88781
Calibrated 10-Fold OOF Accuracy: 0.88850
Optimal Class Multipliers:       [0.98   0.8986 1.    ]

Generated 'submission_90.csv' for submission.
Predicted Class Proportions:
condition
0    0.533
2    0.240
1    0.227
Name: proportion, dtype: float64
